In [23]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [24]:
## load the trained model, scalar pickle , onehot
model = load_model('model.h5')

## load the encode and scalar
with open('onehot_encoder_geo.pkl' , 'rb') as file:
  onehot_encoder_geo = pickle.load(file)
  
with open('label_encoder_gender.pkl' , 'rb') as file:
  label_encoder_gender = pickle.load(file)
  
with open('scaler.pkl' , 'rb') as file:
  scaler = pickle.load(file)

In [25]:
## Example input data
input_data = {
  'CreditScore':600,
  'Geography':'France',
  'Gender':'Male',
  'Age':40,
  'Tenure':3,
  'Balance':60000,
  'NumOfProducts':2,
  'HasCrCard':1,
  'IsActiveMember':1,
  'EstimatedSalary':50000
  }

In [26]:
from sklearn.preprocessing import LabelEncoder , StandardScaler

In [27]:
Label_enc_gen = LabelEncoder()
input_data['Gender'] = label_encoder_gender.transform([input_data['Gender']])[0]

In [28]:
input_data

{'CreditScore': 600,
 'Geography': 'France',
 'Gender': 1,
 'Age': 40,
 'Tenure': 3,
 'Balance': 60000,
 'NumOfProducts': 2,
 'HasCrCard': 1,
 'IsActiveMember': 1,
 'EstimatedSalary': 50000}

In [29]:
geo_encoded = onehot_encoder_geo.transform(
    [[input_data['Geography']]]
).toarray()

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

print(geo_encoded_df)

   Geography_France  Geography_Germany  Geography_Spain
0               1.0                0.0              0.0


c:\Users\Aryan Kumar\miniconda3\envs\genai\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [30]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [31]:
input_df = pd.DataFrame([input_data])

input_df = pd.concat(
    [input_df.drop('Geography', axis=1), geo_encoded_df],
    axis=1
)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [32]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [33]:
prediction = model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 180ms/step


array([[0.0334868]], dtype=float32)

In [34]:
prediction_probs = prediction[0][0]

In [35]:
prediction_probs

0.033486802

In [37]:
if prediction_probs > 0.5:
  print('The customer is likely to churn.')
else:
  print('The Customer is not likely to churn')

The Customer is not likely to churn
